# Language Modelling 2025–2026
## Word Embeddings - Lab Assignment
Students: Azaliia Agisheva, Hryhorii Samofatov


Neural models for word vector representations.
This notebook trains CBOW-style embeddings on English news corpora, evaluates them
qualitatively (cosine similarity, t-SNE) and quantitatively (Reuters classification).

## 0. Setup

Environment assertion, imports, reproducibility seeds, DEBUG mode, global configuration,
artifact directories, and shared I/O helpers. All project code is in this notebook.

In [1]:
import csv
import json
import os
import random
import re
import sys
import unicodedata
import warnings
from pathlib import Path


def configure_gpu_library_path() -> None:
    """WSL2 + pip TensorFlow: set LD_LIBRARY_PATH before importing tensorflow."""
    paths: list[str] = []
    wsl_cuda = Path("/usr/lib/wsl/lib")
    if wsl_cuda.is_dir():
        paths.append(str(wsl_cuda))
    site_packages = Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    nvidia_subdirs = [
        "nvidia/cudnn/lib",
        "nvidia/cublas/lib",
        "nvidia/cuda_runtime/lib",
        "nvidia/cusparse/lib",
        "nvidia/cusolver/lib",
        "nvidia/cufft/lib",
        "nvidia/nccl/lib",
        "nvidia/nvjitlink/lib",
        "nvidia/curand/lib",
        "nvidia/cuda_cupti/lib",
        "nvidia/cuda_nvrtc/lib",
    ]
    for sub in nvidia_subdirs:
        lib_dir = site_packages / sub
        if lib_dir.is_dir():
            paths.append(str(lib_dir))
    if paths:
        existing = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = ":".join(paths + ([existing] if existing else []))


configure_gpu_library_path()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Embedding,
    GlobalMaxPooling1D,
    Input,
    Lambda,
    TextVectorization,
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import reuters
from tensorflow.keras.callbacks import (
    CSVLogger,
    EarlyStopping,
    ReduceLROnPlateau,
)
from tensorflow.keras.layers import Conv1D, MaxPooling1D
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# --- Paths (project root = parent of this notebook) ---
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "eng_news_2024"
CORPUS_30K = DATA_DIR / "eng_news_2024_30k_sentences.txt"
CORPUS_100K = DATA_DIR / "eng_news_2024_100K-sentences.txt"
TARGET_WORDS_PATH = PROJECT_ROOT / "target_words.txt"

ARTIFACTS = PROJECT_ROOT / "artifacts"
DIR_EMBEDDINGS = ARTIFACTS / "embeddings"
DIR_MODELS = ARTIFACTS / "models"
DIR_TOKENIZERS = ARTIFACTS / "tokenizers"
DIR_PLOTS = ARTIFACTS / "plots"
DIR_TABLES = ARTIFACTS / "tables"

for d in (DIR_EMBEDDINGS, DIR_MODELS, DIR_TOKENIZERS, DIR_PLOTS, DIR_TABLES):
    d.mkdir(parents=True, exist_ok=True)

# --- Conda environment (LM) ---
conda_env = os.environ.get("CONDA_DEFAULT_ENV", "")
if conda_env != "LM":
    print(
        f"WARNING: expected conda env 'LM', got {conda_env!r}. "
        "Activate with: conda activate LM"
    )
else:
    print(f"Conda environment: {conda_env}")

print(f"TensorFlow {tf.__version__}")
print("Devices:", tf.config.list_physical_devices())

# --- DEBUG mode (quick end-to-end smoke test) ---
DEBUG = False  # set False for full experiments

if DEBUG:
    DEBUG_N_SENTENCES = 1_000
    MAX_VOCAB_SIZE = 2_000
    WINDOW_SIZE = 2
    EMBEDDING_DIM = 32
    HIDDEN_DIM = 64
    BATCH_SIZE = 128
    EPOCHS = 1
    REUTERS_MAX_LEN = 64
    REUTERS_EPOCHS = 1
    REUTERS_EARLY_STOP_PATIENCE = 1
else:
    DEBUG_N_SENTENCES = None
    MAX_VOCAB_SIZE = 30_000
    WINDOW_SIZE = 2
    EMBEDDING_DIM = 100
    HIDDEN_DIM = 128
    BATCH_SIZE = 128
    EPOCHS = 10
    REUTERS_MAX_LEN = 256
    REUTERS_EPOCHS = 10
    REUTERS_EARLY_STOP_PATIENCE = 2

REUTERS_VOCAB_SIZE = 20_000
RANDOM_SEED = 42
VAL_FRACTION = 0.1
MIN_SENT_TOKENS = 5

CONFIG = {
    "debug": DEBUG,
    "debug_n_sentences": DEBUG_N_SENTENCES,
    "max_vocab_size": MAX_VOCAB_SIZE,
    "window_size": WINDOW_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "reuters_max_len": REUTERS_MAX_LEN,
    "reuters_epochs": REUTERS_EPOCHS,
    "reuters_early_stop_patience": REUTERS_EARLY_STOP_PATIENCE,
    "reuters_vocab_size": REUTERS_VOCAB_SIZE,
    "random_seed": RANDOM_SEED,
    "val_fraction": VAL_FRACTION,
    "min_sent_tokens": MIN_SENT_TOKENS,
}
print("CONFIG:", json.dumps(CONFIG, indent=2))

# --- Reproducibility ---
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# --- Plot style ---
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100


def save_fig(name: str, close: bool = True) -> Path:
    """Save the current matplotlib figure to artifacts/plots/."""
    path = DIR_PLOTS / name
    if not name.endswith(".png"):
        path = path.with_suffix(".png")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    if close:
        plt.close()
    print(f"Saved figure: {path}")
    return path


def append_row(csv_path: Path, row: dict, fieldnames: list[str] | None = None) -> None:
    """Append one row to a CSV, creating the file and header if needed."""
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not csv_path.exists() or csv_path.stat().st_size == 0
    if fieldnames is None:
        fieldnames = list(row.keys())
    with csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def make_run_id(corpus: str, window: int, emb_dim: int, epochs: int, vocab: int) -> str:
    return f"cbow_{corpus}_win{window}_emb{emb_dim}_ep{epochs}_vocab{vocab}"


print("Setup complete.")

I0000 00:00:1779303422.750316  139530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779303422.798728  139530 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Conda environment: LM
TensorFlow 2.21.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CONFIG: {
  "debug": false,
  "debug_n_sentences": null,
  "max_vocab_size": 30000,
  "window_size": 2,
  "embedding_dim": 100,
  "hidden_dim": 128,
  "batch_size": 128,
  "epochs": 10,
  "reuters_max_len": 256,
  "reuters_epochs": 10,
  "reuters_early_stop_patience": 2,
  "reuters_vocab_size": 20000,
  "random_seed": 42,
  "val_fraction": 0.1,
  "min_sent_tokens": 5
}
Setup complete.


I0000 00:00:1779303423.645772  139530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. Data Loading

Load the 30K and 100K English news corpora, compute summary statistics, and apply
DEBUG subsampling when training on a selected corpus.

In [ ]:
def load_corpus(path: Path | str, max_sentences: int | None = None) -> list[str]:
    """Load sentences from a corpus file (30K plain lines or 100K idx\\tsentence)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Corpus not found: {path}")

    sentences: list[str] = []
    has_index: bool | None = None

    with path.open(encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.lstrip("\ufeff").strip()
            if not line:
                continue

            if has_index is None:
                parts = line.split("\t", 1)
                has_index = len(parts) == 2 and parts[0].strip().isdigit()

            if has_index:
                parts = line.split("\t", 1)
                text = parts[1].strip() if len(parts) > 1 else ""
            else:
                text = line

            if text:
                sentences.append(text)
            if max_sentences is not None and len(sentences) >= max_sentences:
                break

    return sentences


def subsample_corpus(
    sentences: list[str],
    debug: bool = DEBUG,
    n: int | None = DEBUG_N_SENTENCES,
) -> list[str]:
    """Return the first *n* sentences when DEBUG mode is enabled."""
    if debug and n is not None:
        return sentences[:n]
    return sentences


def compute_corpus_stats(sentences: list[str], corpus_name: str, subset: str = "full") -> dict:
    """Compute corpus statistics for reporting."""
    n = len(sentences)
    if n == 0:
        return {
            "corpus": corpus_name,
            "subset": subset,
            "n_sentences": 0,
            "avg_chars": 0.0,
            "avg_whitespace_tokens": 0.0,
            "unique_whitespace_tokens": 0,
            "debug": DEBUG,
        }

    char_lens = np.array([len(s) for s in sentences], dtype=np.float64)
    token_lens = np.array([len(s.split()) for s in sentences], dtype=np.float64)
    unique_tokens: set[str] = set()
    for s in sentences:
        unique_tokens.update(s.split())

    return {
        "corpus": corpus_name,
        "subset": subset,
        "n_sentences": n,
        "avg_chars": float(char_lens.mean()),
        "avg_whitespace_tokens": float(token_lens.mean()),
        "unique_whitespace_tokens": len(unique_tokens),
        "debug": DEBUG,
    }


def print_corpus_stats(stats: dict) -> None:
    print(
        f"[{stats['corpus']} / {stats['subset']}] "
        f"sentences={stats['n_sentences']:,} | "
        f"avg_chars={stats['avg_chars']:.1f} | "
        f"avg_tokens={stats['avg_whitespace_tokens']:.1f} | "
        f"unique_tokens={stats['unique_whitespace_tokens']:,}"
    )


# --- Load both corpora (full files) ---
sentences_30k_full = load_corpus(CORPUS_30K)
sentences_100k_full = load_corpus(CORPUS_100K)

# DEBUG subsamples (used later when training on a selected corpus)
sentences_30k = subsample_corpus(sentences_30k_full)
sentences_100k = subsample_corpus(sentences_100k_full)

# Active corpus for downstream stages (toggle here: "30k" or "100k")
ACTIVE_CORPUS = "30k"
sentences = sentences_30k if ACTIVE_CORPUS == "30k" else sentences_100k

# --- Statistics ---
corpus_stats_rows: list[dict] = []
for name, full, subsampled in [
    ("30k", sentences_30k_full, sentences_30k),
    ("100k", sentences_100k_full, sentences_100k),
]:
    for subset_label, data in [("full", full), ("active_subset", subsampled)]:
        stats = compute_corpus_stats(data, name, subset=subset_label)
        print_corpus_stats(stats)
        corpus_stats_rows.append(stats)

corpus_stats_df = pd.DataFrame(corpus_stats_rows)
corpus_stats_path = DIR_TABLES / "corpus_stats.csv"
corpus_stats_df.to_csv(corpus_stats_path, index=False)
print(f"\nSaved: {corpus_stats_path}")
display(corpus_stats_df)

# Sanity checks: format detection
assert len(sentences_30k_full) > 25_000, "30K corpus seems too small"
assert len(sentences_100k_full) > 95_000, "100K corpus seems too small"
assert "\t" not in sentences_30k_full[0] or not sentences_30k_full[0].split("\t")[0].isdigit()
first_100k = sentences_100k_full[0]
print(f"\nSample 30K: {sentences_30k_full[0][:120]}...")
print(f"Sample 100K: {first_100k[:120]}...")
print(f"Active corpus: {ACTIVE_CORPUS} ({len(sentences):,} sentences for training pipeline)")

[30k / full] sentences=30,000 | avg_chars=116.6 | avg_tokens=19.7 | unique_tokens=79,520
[30k / active_subset] sentences=30,000 | avg_chars=116.6 | avg_tokens=19.7 | unique_tokens=79,520
[100k / full] sentences=100,000 | avg_chars=117.0 | avg_tokens=19.8 | unique_tokens=172,454
[100k / active_subset] sentences=100,000 | avg_chars=117.0 | avg_tokens=19.8 | unique_tokens=172,454

Saved: /home/azaliia/projects/LM/practice/artifacts/tables/corpus_stats.csv


,corpus,subset,n_sentences,avg_chars,avg_whitespace_tokens,unique_whitespace_tokens,debug
0,30k,full,30000,116.620667,19.699567,79520,False
1,30k,active_subset,30000,116.620667,19.699567,79520,False
2,100k,full,100000,116.967910,19.753540,172454,False
3,100k,active_subset,100000,116.967910,19.753540,172454,False



Sample 30K: $10-$15 suggested donation goes to Falmouth Land Trust....
Sample 100K: "$100 upfront to process your application," one post read....
Active corpus: 30k (30,000 sentences for training pipeline)


## 2. Text Preprocessing

Normalize raw sentences (unicode, casing, numbers, URLs, punctuation) and drop
sentences shorter than `MIN_SENT_TOKENS` before tokenization.

In [3]:
# Regex patterns for normalization
_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_EMAIL_RE = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")
_NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")
# Keep apostrophes inside contractions (e.g. don't); strip other punctuation
_APOS_BOUNDARY_RE = re.compile(r"(?<!\w)'(?!\w)|(?<!\w)'(?=\w)|(?<=\w)'(?!\w)")
_NONWORD_PUNCT_RE = re.compile(r"[^\w\s']")
_WHITESPACE_RE = re.compile(r"\s+")


def clean_sentence(s: str) -> str:
    """Normalize a single sentence for whitespace tokenization."""
    text = unicodedata.normalize("NFKC", s)
    text = text.lower()
    text = _URL_RE.sub(" ", text)
    text = _EMAIL_RE.sub(" ", text)
    text = _NUM_RE.sub("<num>", text)
    text = _APOS_BOUNDARY_RE.sub(" ", text)
    text = _NONWORD_PUNCT_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def clean_corpus(
    raw_sentences: list[str],
    min_tokens: int = MIN_SENT_TOKENS,
) -> list[str]:
    """Clean all sentences and drop those with fewer than *min_tokens* tokens."""
    cleaned: list[str] = []
    dropped_short = 0
    dropped_empty = 0
    for raw in raw_sentences:
        text = clean_sentence(raw)
        if not text:
            dropped_empty += 1
            continue
        if len(text.split()) < min_tokens:
            dropped_short += 1
            continue
        cleaned.append(text)
    print(
        f"Cleaning: {len(raw_sentences):,} in → {len(cleaned):,} kept | "
        f"dropped empty={dropped_empty:,}, short (<{min_tokens} tokens)={dropped_short:,}"
    )
    return cleaned


# Clean the active corpus (in memory only)
raw_sentences = sentences
cleaned_sentences = clean_corpus(raw_sentences)

# Random before → after examples (reproducible)
rng = random.Random(RANDOM_SEED)
sample_indices = rng.sample(range(len(raw_sentences)), k=min(5, len(raw_sentences)))
print("\n--- 5 random before → after examples ---")
for idx in sample_indices:
    before = raw_sentences[idx]
    after = clean_sentence(before)
    print(f"\n[{idx}] BEFORE: {before[:200]}{'...' if len(before) > 200 else ''}")
    print(f"    AFTER:  {after[:200]}{'...' if len(after) > 200 else ''}")

print(f"\nActive corpus '{ACTIVE_CORPUS}': {len(cleaned_sentences):,} cleaned sentences ready for tokenization.")

Cleaning: 30,000 in → 29,687 kept | dropped empty=0, short (<5 tokens)=313

--- 5 random before → after examples ---

[20952] BEFORE: The American singer has finally spoken out about the attack saying that she is “completely in shock”.
    AFTER:  the american singer has finally spoken out about the attack saying that she is completely in shock

[3648] BEFORE: Brighton 3-2 Tottenham PLAYER RATINGS: Who was the 'worst player on the pitch in the first half but turned the game on its head in the second'?
    AFTER:  brighton num num tottenham player ratings who was the worst player on the pitch in the first half but turned the game on its head in the second

[819] BEFORE: Alabama clawed its way back to tie the game in the top of the sixth with a solo home run from Hodo.
    AFTER:  alabama clawed its way back to tie the game in the top of the sixth with a solo home run from hodo

[24299] BEFORE: The results and the South Tees Clean Air Quality Strategy, which aims to continue making impro

## 3. Tokenization & Vocabulary

Build vocabulary with `TextVectorization`, save token mappings, and check target-word coverage.

In [4]:
# Vocabulary size depends on corpus and DEBUG mode
if DEBUG:
    VOCAB_MAX_TOKENS = MAX_VOCAB_SIZE
else:
    VOCAB_MAX_TOKENS = 50_000 if ACTIVE_CORPUS == "100k" else 30_000

print(f"Building vocabulary (max_tokens={VOCAB_MAX_TOKENS:,}) on corpus '{ACTIVE_CORPUS}'...")

vectorizer = TextVectorization(
    max_tokens=VOCAB_MAX_TOKENS,
    output_mode="int",
    standardize=None,  # already normalized in clean_sentence
    split="whitespace",
    output_sequence_length=None,
)
vectorizer.adapt(cleaned_sentences)

vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)
word_index = {word: idx for idx, word in enumerate(vocab)}
index_word = {idx: word for word, idx in word_index.items()}

print(f"Vocabulary size: {vocab_size:,}")
print(f"Reserved tokens: id 0 = {vocab[0]!r}, id 1 = {vocab[1]!r}")
assert vocab[0] == "", "Expected index 0 to be padding token ''"
assert vocab[1] in ("[UNK]", "[UNK]"), (
    f"Expected index 1 to be UNK token, got {vocab[1]!r}"
)
UNK_TOKEN = vocab[1]

# Save vocabulary for reuse
vocab_path = DIR_TOKENIZERS / f"{ACTIVE_CORPUS}_vocab.json"
vocab_payload = {
    "corpus": ACTIVE_CORPUS,
    "max_tokens": VOCAB_MAX_TOKENS,
    "vocab_size": vocab_size,
    "unk_token": UNK_TOKEN,
    "word_index": word_index,
}
with vocab_path.open("w", encoding="utf-8") as f:
    json.dump(vocab_payload, f, ensure_ascii=False)
print(f"Saved vocabulary: {vocab_path}")

# Target words OOV check
target_words = [
    line.strip()
    for line in TARGET_WORDS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
target_in_vocab = [w for w in target_words if w in word_index]
target_oov = [w for w in target_words if w not in word_index]

target_vocab_df = pd.DataFrame(
    {
        "word": target_words,
        "in_vocab": [w in word_index for w in target_words],
        "token_id": [word_index.get(w, UNK_TOKEN) for w in target_words],
    }
)
target_vocab_path = DIR_TABLES / f"target_words_{ACTIVE_CORPUS}.csv"
target_vocab_df.to_csv(target_vocab_path, index=False)

print(f"\nTarget words: {len(target_in_vocab)}/{len(target_words)} in vocabulary")
if target_oov:
    print(f"OOV ({len(target_oov)}): {', '.join(target_oov)}")
    print("(Excluded from cosine similarity / t-SNE in later stages.)")
else:
    print("All target words are in the vocabulary.")

display(target_vocab_df.head(10))

# Example tokenization
example = cleaned_sentences[0]
example_ids = vectorizer([example]).numpy()[0]
print(f"\nExample sentence: {example[:120]}...")
print(f"Token IDs (first 20): {example_ids[:20].tolist()}")
print(f"Decoded (first 20): {[index_word.get(int(i), UNK_TOKEN) for i in example_ids[:20]]}")

Building vocabulary (max_tokens=30,000) on corpus '30k'...
Vocabulary size: 30,000
Reserved tokens: id 0 = '', id 1 = '[UNK]'


I0000 00:00:1779303425.224820  139530 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9509 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Saved vocabulary: /home/azaliia/projects/LM/practice/artifacts/tokenizers/30k_vocab.json

Target words: 60/60 in vocabulary
All target words are in the vocabulary.


,word,in_vocab,token_id
0,everyone,True,493
1,music,True,496
2,running,True,598
3,worse,True,1959
4,friday,True,211
5,september,True,517
6,someone,True,711
7,getting,True,405
8,reported,True,415
9,expected,True,427



Example sentence: num num suggested donation goes to falmouth land trust...
Token IDs (first 20): [7, 7, 1651, 5126, 813, 3, 10509, 712, 612]
Decoded (first 20): [np.str_('num'), np.str_('num'), np.str_('suggested'), np.str_('donation'), np.str_('goes'), np.str_('to'), np.str_('falmouth'), np.str_('land'), np.str_('trust')]


## 4. Context–Target Sample Generation

Build CBOW training pairs: predict the target word from `2n` context words (`n` left + `n` right).

In [5]:
PAD_UNK_MAX_ID = 1  # skip padding (0) and UNK (1) as target; optional context filter
SKIP_HIGH_OOV_CONTEXT = True
OOV_CONTEXT_FRACTION = 0.5


def sentences_to_id_sequences(
    sentences: list[str],
    vec: TextVectorization,
) -> list[np.ndarray]:
    """Convert cleaned sentences to variable-length int32 token-id arrays."""
    id_seqs: list[np.ndarray] = []
    for s in tqdm(sentences, desc="Tokenizing sentences"):
        ids = vec([s]).numpy()[0].astype(np.int32)
        id_seqs.append(ids)
    return id_seqs


def count_cbow_samples(
    id_seqs: list[np.ndarray],
    n: int,
    skip_high_oov: bool = SKIP_HIGH_OOV_CONTEXT,
    oov_fraction: float = OOV_CONTEXT_FRACTION,
) -> int:
    """Count CBOW samples without materializing arrays."""
    total = 0
    ctx_len = 2 * n
    for ids in id_seqs:
        L = len(ids)
        if L < ctx_len + 1:
            continue
        for i in range(n, L - n):
            if ids[i] <= PAD_UNK_MAX_ID:
                continue
            ctx = np.concatenate((ids[i - n : i], ids[i + 1 : i + n + 1]))
            if skip_high_oov and np.mean(ctx <= PAD_UNK_MAX_ID) > oov_fraction:
                continue
            total += 1
    return total


def generate_samples(
    id_seqs: list[np.ndarray],
    n: int,
    skip_high_oov: bool = SKIP_HIGH_OOV_CONTEXT,
    oov_fraction: float = OOV_CONTEXT_FRACTION,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate CBOW (context, target) pairs.

    Returns
    -------
    X : (N, 2n) int32 context token ids
    y : (N,) int32 target token ids
    """
    ctx_len = 2 * n
    n_samples = count_cbow_samples(id_seqs, n, skip_high_oov, oov_fraction)
    X = np.empty((n_samples, ctx_len), dtype=np.int32)
    y = np.empty(n_samples, dtype=np.int32)

    idx = 0
    for ids in id_seqs:
        L = len(ids)
        if L < ctx_len + 1:
            continue
        for i in range(n, L - n):
            target = ids[i]
            if target <= PAD_UNK_MAX_ID:
                continue
            ctx = np.concatenate((ids[i - n : i], ids[i + 1 : i + n + 1]))
            if skip_high_oov and np.mean(ctx <= PAD_UNK_MAX_ID) > oov_fraction:
                continue
            X[idx] = ctx
            y[idx] = target
            idx += 1

    assert idx == n_samples, f"Filled {idx} samples, expected {n_samples}"
    return X, y


def decode_sample(
    context_ids: np.ndarray,
    target_id: int,
    index_word: dict[int, str],
) -> tuple[list[str], str]:
    ctx_words = [index_word.get(int(t), UNK_TOKEN) for t in context_ids]
    target_word = index_word.get(int(target_id), UNK_TOKEN)
    return ctx_words, target_word


def print_sample_examples(
    X: np.ndarray,
    y: np.ndarray,
    index_word: dict[int, str],
    n_examples: int = 3,
    seed: int = RANDOM_SEED,
) -> None:
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(y), size=min(n_examples, len(y)), replace=False)
    print(f"\n--- {len(indices)} example (context → target) pairs ---")
    for k, j in enumerate(indices):
        ctx_words, target_word = decode_sample(X[j], y[j], index_word)
        ctx_str = " ".join(ctx_words)
        print(f"[{k + 1}] context: [{ctx_str}]  →  target: '{target_word}'")


# Tokenize cleaned corpus once
sentence_id_seqs = sentences_to_id_sequences(cleaned_sentences, vectorizer)

# Shape checks for n ∈ {2, 5, 8} on a small prefix (fast in DEBUG)
_shape_check_seqs = sentence_id_seqs[: min(50, len(sentence_id_seqs))]
print("Shape checks (first 50 sentences):")
for n_test in (2, 5, 8):
    X_test, y_test = generate_samples(_shape_check_seqs, n_test)
    assert X_test.shape == (len(y_test), 2 * n_test), (
        f"n={n_test}: expected X shape (_, {2 * n_test}), got {X_test.shape}"
    )
    assert y_test.ndim == 1, f"n={n_test}: y must be 1-D"
    print(f"  n={n_test}: X{X_test.shape}, y{y_test.shape} ({len(y_test):,} samples)")

# Main training window from CONFIG
n = WINDOW_SIZE
X, y = generate_samples(sentence_id_seqs, n)

mem_mb = (X.nbytes + y.nbytes) / (1024 ** 2)
print(
    f"\nGenerated {len(y):,} samples | window n={n} | context dim={2 * n} | "
    f"memory ≈ {mem_mb:.1f} MiB"
)

# Train / validation split (90% / 10%)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=VAL_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
)
print(
    f"Train: {len(y_train):,} samples | Val: {len(y_val):,} samples "
    f"({VAL_FRACTION:.0%} hold-out)"
)

print_sample_examples(X_train, y_train, index_word, n_examples=3)

# Persist summary for the report
samples_summary = {
    "corpus": ACTIVE_CORPUS,
    "window_size": n,
    "n_samples": int(len(y)),
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "context_dim": 2 * n,
    "memory_mb": round(mem_mb, 2),
    "skip_high_oov_context": SKIP_HIGH_OOV_CONTEXT,
    "debug": DEBUG,
}
samples_summary_path = DIR_TABLES / f"cbow_samples_{ACTIVE_CORPUS}_win{n}.json"
with samples_summary_path.open("w", encoding="utf-8") as f:
    json.dump(samples_summary, f, indent=2)
print(f"Saved: {samples_summary_path}")

Tokenizing sentences:   0%|          | 0/29687 [00:00<?, ?it/s]

Shape checks (first 50 sentences):
  n=2: X(760, 4), y(760,) (760 samples)
  n=5: X(475, 10), y(475,) (475 samples)
  n=8: X(257, 16), y(257,) (257 samples)

Generated 478,154 samples | window n=2 | context dim=4 | memory ≈ 9.1 MiB
Train: 430,338 samples | Val: 47,816 samples (10% hold-out)

--- 3 example (context → target) pairs ---
[1] context: [prison with lengthy waitlist]  →  target: 'a'
[2] context: [region as iran aligned]  →  target: 'the'
[3] context: [independent tools libraries to]  →  target: 'and'
Saved: /home/azaliia/projects/LM/practice/artifacts/tables/cbow_samples_30k_win2.json


## 5. Model Architecture

CBOW model: embed context words → average → dense hidden → softmax over vocabulary.

In [ ]:
def build_cbow_model(
    vocab_size: int,
    embedding_dim: int,
    window_size: int,
    hidden_dim: int,
    learning_rate: float = 1e-3,
) -> Model:
    """
    Build and compile a CBOW word-prediction model (Functional API).

    Input: (batch, 2*window_size) context token ids
    Output: (batch, vocab_size) softmax over target word
    """
    context_len = 2 * window_size
    inputs = Input(shape=(context_len,), dtype="int32", name="context")
    embedded = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        name="embeddings",
    )(inputs)
    averaged = Lambda(
        lambda x: K.mean(x, axis=1),
        name="context_mean",
    )(embedded)
    hidden = Dense(hidden_dim, activation="relu", name="hidden")(averaged)
    outputs = Dense(vocab_size, activation="softmax", name="target")(hidden)

    model = Model(inputs=inputs, outputs=outputs, name="cbow")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "sparse_categorical_accuracy",
            keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc"),
        ],
    )
    return model


def extract_embedding_matrix(model: Model) -> np.ndarray:
    """Return the learned embedding matrix from the 'embeddings' layer."""
    return model.get_layer("embeddings").get_weights()[0]


In [7]:
import time

EXPERIMENTS_CSV_PATH = DIR_TABLES / "experiments.csv"
EXPERIMENT_FIELDNAMES = [
    "run_id",
    "corpus",
    "n_sentences",
    "window_size",
    "embedding_dim",
    "vocab_size",
    "epochs",
    "batch_size",
    "train_loss",
    "val_loss",
    "val_top1_acc",
    "val_top5_acc",
    "train_time_s",
]

# Full experiment grid (plan §10.2). In DEBUG mode only #1 is executed.
CBOW_EXPERIMENT_GRID: list[dict] = [
    {"id": 1, "corpus": "30k", "window_size": 2, "embedding_dim": 100, "epochs": 5},
    {"id": 2, "corpus": "30k", "window_size": 5, "embedding_dim": 100, "epochs": 5},
    {"id": 3, "corpus": "30k", "window_size": 2, "embedding_dim": 300, "epochs": 5},
    {"id": 4, "corpus": "100k", "window_size": 2, "embedding_dim": 100, "epochs": 5},
    {"id": 5, "corpus": "100k", "window_size": 5, "embedding_dim": 100, "epochs": 5},
    {"id": 6, "corpus": "100k", "window_size": 2, "embedding_dim": 300, "epochs": 5},
    {"id": 7, "corpus": "100k", "window_size": 5, "embedding_dim": 100, "epochs": 10},
    {"id": 8, "corpus": "100k", "window_size": 8, "embedding_dim": 100, "epochs": 5},
]


def resolve_experiment_cfg(cfg: dict) -> dict:
    """Merge grid entry with DEBUG / full-run defaults."""
    c = dict(cfg)
    corpus = c["corpus"]
    if DEBUG:
        c["max_vocab_size"] = MAX_VOCAB_SIZE
        c["hidden_dim"] = HIDDEN_DIM
        c["batch_size"] = BATCH_SIZE
        c["embedding_dim"] = EMBEDDING_DIM
        c["epochs"] = EPOCHS
        c["debug_subsample"] = True
    else:
        c.setdefault("max_vocab_size", 50_000 if corpus == "100k" else 30_000)
        c.setdefault("hidden_dim", 128)
        c.setdefault("batch_size", 256)
        c.setdefault("debug_subsample", False)
    return c


def plot_training_history(history: keras.callbacks.History, run_id: str) -> Path:
    """Plot loss and accuracy curves; save to artifacts/plots/."""
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(h["loss"], label="train")
    if "val_loss" in h:
        axes[0].plot(h["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    if "sparse_categorical_accuracy" in h:
        axes[1].plot(h["sparse_categorical_accuracy"], label="train top-1")
        if "val_sparse_categorical_accuracy" in h:
            axes[1].plot(h["val_sparse_categorical_accuracy"], label="val top-1")
    if "top5_acc" in h:
        axes[1].plot(h["top5_acc"], label="train top-5", linestyle="--")
        if "val_top5_acc" in h:
            axes[1].plot(h["val_top5_acc"], label="val top-5", linestyle="--")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    return save_fig(f"{run_id}_loss")


def run_experiment(cfg: dict) -> dict:
    """
    End-to-end CBOW experiment: load → preprocess → tokenize → samples → train → save.

    Returns a dict with experiment_row, run_id, model, embeddings, and vocab mappings.
    """
    cfg = resolve_experiment_cfg(cfg)
    corpus = cfg["corpus"]
    window_size = cfg["window_size"]
    embedding_dim = cfg["embedding_dim"]
    epochs = cfg["epochs"]
    max_vocab = cfg["max_vocab_size"]
    hidden_dim = cfg["hidden_dim"]
    batch_size = cfg["batch_size"]
    debug_subsample = cfg.get("debug_subsample", DEBUG)

    corpus_path = CORPUS_30K if corpus == "30k" else CORPUS_100K
    print(f"\n{'=' * 60}\nExperiment corpus={corpus}, n={window_size}, d={embedding_dim}, epochs={epochs}\n{'=' * 60}")

    raw = load_corpus(corpus_path)
    if debug_subsample and DEBUG_N_SENTENCES is not None:
        raw = raw[:DEBUG_N_SENTENCES]
    cleaned = clean_corpus(raw, min_tokens=MIN_SENT_TOKENS)

    vec = TextVectorization(
        max_tokens=max_vocab,
        output_mode="int",
        standardize=None,
        split="whitespace",
        output_sequence_length=None,
    )
    vec.adapt(cleaned)
    vocab = vec.get_vocabulary()
    v_size = len(vocab)
    w_index = {word: idx for idx, word in enumerate(vocab)}
    i_word = {idx: word for word, idx in w_index.items()}

    id_seqs = sentences_to_id_sequences(cleaned, vec)
    X_all, y_all = generate_samples(id_seqs, window_size)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_all,
        y_all,
        test_size=VAL_FRACTION,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    run_id = make_run_id(corpus, window_size, embedding_dim, epochs, v_size)
    keras.backend.clear_session()
    model = build_cbow_model(v_size, embedding_dim, window_size, hidden_dim)

    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
        .shuffle(len(y_tr), seed=RANDOM_SEED)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_va, y_va))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    cb = [
        EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", patience=1, factor=0.5, verbose=1),
        CSVLogger(str(DIR_TABLES / f"training_log_{run_id}.csv")),
    ]

    t0 = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cb, verbose=1)
    train_time_s = time.perf_counter() - t0

    plot_training_history(history, run_id)
    E = extract_embedding_matrix(model)
    np.save(DIR_EMBEDDINGS / f"{run_id}_E.npy", E)
    with (DIR_EMBEDDINGS / f"{run_id}_word_index.json").open("w", encoding="utf-8") as f:
        json.dump(w_index, f, ensure_ascii=False)
    model.save(DIR_MODELS / f"{run_id}.keras")

    hist = history.history
    experiment_row = {
        "run_id": run_id,
        "corpus": corpus,
        "n_sentences": len(cleaned),
        "window_size": window_size,
        "embedding_dim": embedding_dim,
        "vocab_size": v_size,
        "epochs": len(hist.get("loss", [])),
        "batch_size": batch_size,
        "train_loss": float(hist["loss"][-1]),
        "val_loss": float(min(hist["val_loss"])),
        "val_top1_acc": float(max(hist["val_sparse_categorical_accuracy"])),
        "val_top5_acc": float(max(hist["val_top5_acc"])),
        "train_time_s": round(train_time_s, 2),
    }
    append_row(EXPERIMENTS_CSV_PATH, experiment_row, fieldnames=EXPERIMENT_FIELDNAMES)

    print(f"Done {run_id} in {train_time_s:.1f}s | val_loss={experiment_row['val_loss']:.4f}")
    return {
        "run_id": run_id,
        "experiment_row": experiment_row,
        "model": model,
        "embeddings": E,
        "word_index": w_index,
        "index_word": i_word,
        "vocab_size": v_size,
        "vectorizer": vec,
        "cfg": cfg,
    }

## 6. Experiment Runner

Run the CBOW experiment grid.

In [8]:
def run_experiment_grid(
    grid: list[dict] | None = None,
    *,
    debug_only_first: bool = DEBUG,
    skip_existing: bool = True,
) -> pd.DataFrame:
    """
    Run all configs in the grid and append rows to experiments.csv.

    Parameters
    ----------
    debug_only_first : if True (DEBUG), run only grid entry #1.
    skip_existing : skip runs whose run_id is already in experiments.csv.
    """
    grid = grid or CBOW_EXPERIMENT_GRID
    configs = grid[:1] if debug_only_first else grid

    def _cfg_key(c: dict) -> tuple:
        return (c["corpus"], c["window_size"], c["embedding_dim"], c["epochs"])

    existing_keys: set[tuple] = set()
    if skip_existing and EXPERIMENTS_CSV_PATH.exists():
        df_done = pd.read_csv(EXPERIMENTS_CSV_PATH)
        for _, row in df_done.iterrows():
            existing_keys.add(
                (row["corpus"], int(row["window_size"]), int(row["embedding_dim"]), int(row["epochs"]))
            )

    results: list[dict] = []
    for cfg in configs:
        cfg_resolved = resolve_experiment_cfg(cfg)
        if skip_existing and _cfg_key(cfg_resolved) in existing_keys:
            print(f"Skipping existing config: {_cfg_key(cfg_resolved)}")
            continue

        exp_id = cfg.get("id", "?")
        print(f"\n>>> Grid experiment #{exp_id}")
        out = run_experiment(cfg)
        results.append(out["experiment_row"])
        keras.backend.clear_session()

    df = pd.DataFrame(results)
    print(f"\nCompleted {len(results)} new experiment(s).")
    display(df)
    display(pd.read_csv(EXPERIMENTS_CSV_PATH))
    return df


# Run the grid (DEBUG → #1 only; full mode → all 8 configs)
grid_results_df = run_experiment_grid()

# Keep latest run artifacts in notebook globals for §7–§8
if len(grid_results_df) > 0:
    _latest_rid = grid_results_df.iloc[-1]["run_id"]
    RUN_ID = _latest_rid
    E_trained = np.load(DIR_EMBEDDINGS / f"{RUN_ID}_E.npy")
    with (DIR_EMBEDDINGS / f"{RUN_ID}_word_index.json").open(encoding="utf-8") as f:
        word_index = json.load(f)
    index_word = {idx: word for word, idx in word_index.items()}
    print(f"Latest grid run loaded: {RUN_ID}")


>>> Grid experiment #1

Experiment corpus=30k, n=2, d=100, epochs=5
Cleaning: 30,000 in → 29,687 kept | dropped empty=0, short (<5 tokens)=313


Tokenizing sentences:   0%|          | 0/29687 [00:00<?, ?it/s]

Epoch 1/5


I0000 00:00:1779303538.292551  139931 service.cc:153] XLA service 0x73d2300968a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779303538.292577  139931 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1779303538.307734  139931 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779303538.371204  139931 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1779303538.378009  139931 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2495155__.20
I0000 00:00:1779303538.390374  139931 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints se

  11/1682 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 10.3058 - sparse_categorical_accuracy: 0.0230 - top5_acc: 0.0645

...

I0000 00:00:1779305641.692228  139932 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26467276__.20
I0000 00:00:1779305641.700856  139932 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779305642.004749  139932 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779305642.150801  153848 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1779305643.016201  153828 subprocess_compilation.cc:348] ptxas warning : Registers are spil

2079/2079 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 7.3633 - sparse_categorical_accuracy: 0.0599 - top5_acc: 0.1851 - val_loss: 7.1291 - val_sparse_categorical_accuracy: 0.0643 - val_top5_acc: 0.1981 - learning_rate: 0.0010
Epoch 2/5
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - loss: 6.9630 - sparse_categorical_accuracy: 0.0753 - top5_acc: 0.2053 - val_loss: 7.0190 - val_sparse_categorical_accuracy: 0.0858 - val_top5_acc: 0.2145 - learning_rate: 0.0010
Epoch 3/5
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - loss: 6.7411 - sparse_categorical_accuracy: 0.0963 - top5_acc: 0.2252 - val_loss: 6.9369 - val_sparse_categorical_accuracy: 0.0967 - val_top5_acc: 0.2270 - learning_rate: 0.0010
Epoch 4/5
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 6.5152 - sparse_categorical_accuracy: 0.1141 - top5_acc: 0.2432 - val_loss: 6.8901 - val_sparse_categorical_accuracy: 0.1047 - val_top5_acc: 0.2339 - learning_rate: 0.0010
Epoch 5/5
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - loss: 6.2953 - sp

,run_id,corpus,n_sentences,window_size,embedding_dim,vocab_size,epochs,batch_size,train_loss,val_loss,val_top1_acc,val_top5_acc,train_time_s
0,cbow_30k_win2_emb100_ep5_vocab30000,30k,29687,2,100,30000,5,256,5.686319,6.477578,0.158483,0.297808,60.48
1,cbow_30k_win5_emb100_ep5_vocab30000,30k,29687,5,100,30000,5,256,6.384019,6.987684,0.105866,0.233034,35.24
2,cbow_30k_win2_emb300_ep5_vocab30000,30k,29687,2,300,30000,5,256,5.328689,6.456617,0.162623,0.303978,52.83
3,cbow_100k_win2_emb100_ep5_vocab50000,100k,99065,2,100,50000,5,256,5.141646,6.165301,0.183741,0.335743,246.45
4,cbow_100k_win5_emb100_ep5_vocab50000,100k,99065,5,100,50000,5,256,5.854170,6.623945,0.137074,0.270567,150.32
5,cbow_100k_win2_emb300_ep5_vocab50000,100k,99065,2,300,50000,5,256,4.795797,6.152708,0.187480,0.338889,295.47
6,cbow_100k_win5_emb100_ep10_vocab50000,100k,99065,5,100,50000,6,256,5.585392,6.620987,0.138319,0.271658,185.04
7,cbow_100k_win8_emb100_ep5_vocab50000,100k,99065,8,100,50000,5,256,6.295267,6.885544,0.108549,0.236890,98.03


,run_id,corpus,n_sentences,window_size,embedding_dim,vocab_size,epochs,batch_size,train_loss,val_loss,val_top1_acc,val_top5_acc,train_time_s
0,cbow_30k_win2_emb100_ep5_vocab30000,30k,29687,2,100,30000,5,256,5.686319,6.477578,0.158483,0.297808,60.48
1,cbow_30k_win5_emb100_ep5_vocab30000,30k,29687,5,100,30000,5,256,6.384019,6.987684,0.105866,0.233034,35.24
2,cbow_30k_win2_emb300_ep5_vocab30000,30k,29687,2,300,30000,5,256,5.328689,6.456617,0.162623,0.303978,52.83
3,cbow_100k_win2_emb100_ep5_vocab50000,100k,99065,2,100,50000,5,256,5.141646,6.165301,0.183741,0.335743,246.45
4,cbow_100k_win5_emb100_ep5_vocab50000,100k,99065,5,100,50000,5,256,5.854170,6.623945,0.137074,0.270567,150.32
5,cbow_100k_win2_emb300_ep5_vocab50000,100k,99065,2,300,50000,5,256,4.795797,6.152708,0.187480,0.338889,295.47
6,cbow_100k_win5_emb100_ep10_vocab50000,100k,99065,5,100,50000,6,256,5.585392,6.620987,0.138319,0.271658,185.04
7,cbow_100k_win8_emb100_ep5_vocab50000,100k,99065,8,100,50000,5,256,6.295267,6.885544,0.108549,0.236890,98.03


Latest grid run loaded: cbow_100k_win8_emb100_ep5_vocab50000


## 7. Qualitative Evaluation

Cosine similarity and t-SNE (before/after training) for each CBOW run in `experiments.csv`.

In [9]:
def load_run_artifacts(run_id: str) -> tuple[np.ndarray, dict, dict, int]:
    """Load embedding matrix and vocab mappings for a CBOW run."""
    E = np.load(DIR_EMBEDDINGS / f"{run_id}_E.npy")
    with (DIR_EMBEDDINGS / f"{run_id}_word_index.json").open(encoding="utf-8") as f:
        w_index = json.load(f)
    i_word = {idx: word for word, idx in w_index.items()}
    return E, w_index, i_word, E.shape[1]


def most_similar(
    word: str,
    embeddings: np.ndarray,
    word_index: dict,
    index_word: dict,
    k: int = 10,
) -> list[tuple[str, float]]:
    """Top-k cosine neighbors for *word* (excludes the query word)."""
    if word not in word_index:
        return []
    idx = word_index[word]
    vec = embeddings[idx : idx + 1]
    sims = cosine_similarity(vec, embeddings)[0]
    sims[idx] = -np.inf
    top_idx = np.argsort(sims)[-k:][::-1]
    return [(index_word[i], float(sims[i])) for i in top_idx if i in index_word]


def visualize_tsne_embeddings(
    words: list[str],
    embeddings: np.ndarray,
    word_index: dict,
    filename: str | Path | None = None,
    random_state: int = RANDOM_SEED,
) -> None:
    """t-SNE plot for selected words (from course materials)."""
    words = [w for w in words if w in word_index]
    if len(words) < 2:
        print(f"Need ≥2 in-vocab words for t-SNE, got {len(words)}")
        return
    indices = [word_index[w] for w in words]
    selected = embeddings[indices]
    perplexity = min(30, max(2, len(words) - 1))
    reduced = TSNE(n_components=2, perplexity=perplexity, random_state=random_state).fit_transform(
        selected
    )
    plt.figure(figsize=(10, 10))
    for i, word in enumerate(words):
        plt.scatter(reduced[i, 0], reduced[i, 1])
        plt.annotate(
            word,
            xy=(reduced[i, 0], reduced[i, 1]),
            xytext=(5, 2),
            textcoords="offset points",
            ha="right",
            va="bottom",
        )
    if filename:
        save_fig(Path(filename).stem, close=True)
    else:
        plt.show()


def visualize_all_tsne_embeddings(
    embeddings: np.ndarray,
    word_index: dict,
    words_to_plot: list[str],
    words_to_label: list[str] | None = None,
    filename: str | Path | None = None,
    random_state: int = RANDOM_SEED,
) -> None:
    """t-SNE for many words; label only *words_to_label* (from course materials)."""
    index_word = {idx: w for w, idx in word_index.items()}
    words_to_plot = [w for w in words_to_plot if w in word_index]
    if words_to_label is None:
        words_to_label = words_to_plot
    words_to_label = set(words_to_label).intersection(words_to_plot)
    if len(words_to_plot) < 2:
        print("Need ≥2 words for t-SNE")
        return
    indices = [word_index[w] for w in words_to_plot]
    selected = embeddings[indices]
    perplexity = min(30, max(2, len(words_to_plot) - 1))
    reduced = TSNE(n_components=2, perplexity=perplexity, random_state=random_state).fit_transform(
        selected
    )
    plt.figure(figsize=(12, 12))
    for i, idx in enumerate(indices):
        plt.scatter(reduced[i, 0], reduced[i, 1], alpha=0.5)
        if index_word[idx] in words_to_label:
            plt.annotate(
                index_word[idx],
                xy=(reduced[i, 0], reduced[i, 1]),
                xytext=(5, 2),
                textcoords="offset points",
                ha="right",
                va="bottom",
            )
    if filename:
        save_fig(Path(filename).stem, close=True)
    else:
        plt.show()


def top_frequent_words(word_index: dict, n: int = 500) -> list[str]:
    """Keras vocab: lower indices ≈ higher frequency (skip pad/UNK at 0,1)."""
    pairs = [(idx, w) for w, idx in word_index.items() if idx > 1 and w]
    pairs.sort(key=lambda x: x[0])
    return [w for _, w in pairs[:n]]


def run_qualitative_evaluation(
    run_id: str,
    exp_row: pd.Series | None = None,
    hidden_dim: int = HIDDEN_DIM,
) -> None:
    """Cosine tables + t-SNE before/after + top-500 plot for one CBOW run."""
    E_after, w_index, i_word, emb_dim = load_run_artifacts(run_id)
    if exp_row is None:
        exp_row = experiments_unique.loc[experiments_unique["run_id"] == run_id].iloc[0]
    window_size = int(exp_row["window_size"])
    vocab_size_run = int(exp_row["vocab_size"])

    target_words_list = [
        line.strip()
        for line in TARGET_WORDS_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    in_vocab_targets = [w for w in target_words_list if w in w_index]

    # --- Cosine similarity ---
    cosine_rows: list[dict] = []
    print(f"\n=== Cosine similarity ({run_id}) ===")
    for tw in target_words_list:
        if tw not in w_index:
            print(f"  {tw}: OOV (skipped)")
            continue
        neighbors = most_similar(tw, E_after, w_index, i_word, k=10)
        print(f"  {tw}: {', '.join(f'{w}({s:.3f})' for w, s in neighbors[:5])} ...")
        for rank, (nw, score) in enumerate(neighbors, start=1):
            cosine_rows.append(
                {"run_id": run_id, "target_word": tw, "rank": rank, "neighbor": nw, "cosine": score}
            )
    cosine_path = DIR_TABLES / f"cosine_{run_id}.csv"
    pd.DataFrame(cosine_rows).to_csv(cosine_path, index=False)
    print(f"Saved {cosine_path}")

    # --- t-SNE before (random init) ---
    keras.backend.clear_session()
    m_before = build_cbow_model(vocab_size_run, emb_dim, window_size, hidden_dim)
    E_before = extract_embedding_matrix(m_before)
    del m_before
    keras.backend.clear_session()

    tsne_targets = in_vocab_targets if len(in_vocab_targets) >= 2 else in_vocab_targets + ["the", "and"]
    visualize_tsne_embeddings(
        tsne_targets,
        E_before,
        w_index,
        filename=DIR_PLOTS / f"{run_id}_tsne_before",
        random_state=RANDOM_SEED,
    )
    visualize_tsne_embeddings(
        tsne_targets,
        E_after,
        w_index,
        filename=DIR_PLOTS / f"{run_id}_tsne_after",
        random_state=RANDOM_SEED,
    )

    top500 = top_frequent_words(w_index, n=500)
    visualize_all_tsne_embeddings(
        E_after,
        w_index,
        top500,
        words_to_label=in_vocab_targets,
        filename=DIR_PLOTS / f"{run_id}_tsne_top500",
        random_state=RANDOM_SEED,
    )


experiments_df = pd.read_csv(EXPERIMENTS_CSV_PATH)
experiments_unique = experiments_df.drop_duplicates(subset=["run_id"], keep="last")
print(f"Qualitative evaluation for {len(experiments_unique)} CBOW run(s)")

for _, exp_row in experiments_unique.iterrows():
    run_qualitative_evaluation(exp_row["run_id"], exp_row)

# Best run by validation top-1 accuracy (for report discussion)
best_row = experiments_unique.loc[experiments_unique["val_top1_acc"].idxmax()]
print(
    f"\nBest CBOW run (val top-1): {best_row['run_id']} "
    f"(acc={best_row['val_top1_acc']:.4f}, corpus={best_row['corpus']})"
)

Qualitative evaluation for 8 CBOW run(s)

=== Cosine similarity (cbow_30k_win2_emb100_ep5_vocab30000) ===
  everyone: anyone(0.834), recreational(0.792), tinder(0.784), larcombe(0.779), anybody(0.776) ...
  music: items(0.694), warnings(0.659), guidance(0.657), services(0.652), courses(0.646) ...
  running: happening(0.633), question(0.622), step(0.617), answer(0.614), power(0.605) ...
  worse: squeezed(0.813), happy(0.800), cool(0.790), pleased(0.785), doubled(0.766) ...
  friday: thursday(0.937), monday(0.926), wednesday(0.922), tuesday(0.916), saturday(0.839) ...
  september: february(0.927), january(0.910), december(0.908), june(0.906), october(0.904) ...
  someone: taxing(0.804), telesco(0.801), destined(0.784), gil(0.782), beijing(0.780) ...
  getting: hard(0.814), better(0.786), likely(0.772), very(0.764), difficult(0.763) ...
  reported: till(0.734), announced(0.703), revealed(0.702), denied(0.696), insisting(0.679) ...
  expected: scheduled(0.871), supposed(0.857), going(0.835

## 8. Reuters Classification

Train Reuters CNN classifiers: one random baseline + frozen / fine-tuned pretrained embeddings per CBOW run.

Full mode: `REUTERS_EPOCHS=10`, `REUTERS_MAX_LEN=256`, **EarlyStopping** on `val_loss` (restores best weights) and `val_accuracy`.

In [10]:
from sklearn.metrics import confusion_matrix, f1_score

REUTERS_NUM_CLASSES = 46
REUTERS_BATCH_SIZE = 128 if not DEBUG else 64
REUTERS_RESULTS_PATH = DIR_TABLES / "reuters_results.csv"
REUTERS_RESULT_FIELDS = [
    "run_id",
    "setup",
    "embedding_dim",
    "reuters_vocab_size",
    "coverage_pct",
    "test_loss",
    "test_accuracy",
    "test_macro_f1",
    "train_time_s",
]


def build_reuters_index() -> tuple[dict[int, str], dict[str, int]]:
    """Reuters word_index with Keras +3 offset; inverse id → word."""
    raw_index = reuters.get_word_index()
    index_to_word = {i + 3: w for w, i in raw_index.items()}
    index_to_word[0] = ""
    index_to_word[1] = "<START>"
    index_to_word[2] = "<UNK>"
    word_to_index = {w: i for i, w in index_to_word.items()}
    return index_to_word, word_to_index


def build_pretrained_embedding_matrix(
    E_cbow: np.ndarray,
    cbow_word_index: dict,
    reuters_index_to_word: dict[int, str],
    reuters_vocab_size: int = REUTERS_VOCAB_SIZE,
) -> tuple[np.ndarray, float]:
    """Map CBOW vectors into Reuters embedding matrix; return matrix and coverage %."""
    emb_dim = E_cbow.shape[1]
    matrix = np.random.normal(0, 0.1, (reuters_vocab_size, emb_dim)).astype(np.float32)
    found, total = 0, 0
    for rid in range(3, reuters_vocab_size):
        total += 1
        rw = reuters_index_to_word.get(rid, "")
        if not rw:
            continue
        cw = clean_sentence(rw)
        if cw in cbow_word_index:
            matrix[rid] = E_cbow[cbow_word_index[cw]]
            found += 1
    coverage = 100.0 * found / total if total else 0.0
    return matrix, coverage


def build_reuters_classifier(
    embedding_dim: int,
    embedding_matrix: np.ndarray | None = None,
    trainable_embeddings: bool = True,
    max_len: int = REUTERS_MAX_LEN,
) -> Sequential:
    """CNN text classifier (from course build_model.py)."""
    if embedding_matrix is None:
        emb = Embedding(
            input_dim=REUTERS_VOCAB_SIZE,
            output_dim=embedding_dim,
            input_length=max_len,
        )
    else:
        emb = Embedding(
            input_dim=REUTERS_VOCAB_SIZE,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=trainable_embeddings,
            input_length=max_len,
        )
    model = Sequential(
        [
            emb,
            Conv1D(128, 5, activation="relu"),
            MaxPooling1D(5),
            Conv1D(128, 5, activation="relu"),
            GlobalMaxPooling1D(),
            Dense(128, activation="relu"),
            Dropout(0.5),
            Dense(REUTERS_NUM_CLASSES, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def plot_reuters_training(history: keras.callbacks.History, run_id: str, setup: str) -> Path:
    h = history.history
    plt.figure(figsize=(10, 4))
    plt.plot(h["loss"], label="train")
    plt.plot(h["val_loss"], label="val")
    plt.plot(h["accuracy"], label="train acc")
    plt.plot(h["val_accuracy"], label="val acc")
    plt.xlabel("Epoch")
    plt.legend()
    plt.title(f"Reuters {setup} — {run_id}")
    return save_fig(f"{run_id}_clf_{setup}_curves")


def plot_confusion_top_classes(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    run_id: str,
    setup: str,
    top_n: int = 10,
) -> Path:
    """Confusion matrix for the *top_n* most frequent true classes."""
    counts = np.bincount(y_true, minlength=REUTERS_NUM_CLASSES)
    top_classes = np.argsort(counts)[-top_n:]
    mask = np.isin(y_true, top_classes)
    cm = confusion_matrix(y_true[mask], y_pred[mask], labels=list(top_classes))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=top_classes, yticklabels=top_classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion (top-{top_n} classes) — {setup}")
    return save_fig(f"{run_id}_clf_{setup}_confusion")


def train_and_evaluate_reuters(
    run_id: str,
    setup: str,
    embedding_matrix: np.ndarray | None,
    trainable: bool,
    coverage_pct: float = 0.0,
    skip_if_exists: bool = True,
) -> dict | None:
    """Train one Reuters setup and append results to reuters_results.csv."""
    emb_dim = embedding_matrix.shape[1] if embedding_matrix is not None else 100

    if skip_if_exists and REUTERS_RESULTS_PATH.exists():
        done = pd.read_csv(REUTERS_RESULTS_PATH)
        if ((done["run_id"] == run_id) & (done["setup"] == setup)).any():
            print(f"Skipping existing Reuters: {run_id} / {setup}")
            return None

    print(f"\n--- Reuters {setup} | run_id={run_id} | emb_dim={emb_dim} ---")
    keras.backend.clear_session()
    model = build_reuters_classifier(emb_dim, embedding_matrix, trainable_embeddings=trainable)

    reuters_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=REUTERS_EARLY_STOP_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=REUTERS_EARLY_STOP_PATIENCE,
            restore_best_weights=False,
            verbose=0,
        ),
    ]

    t0 = time.perf_counter()
    history = model.fit(
        X_rt_tr,
        y_rt_train_cat,
        validation_data=(X_rt_val, y_rt_val_cat),
        epochs=REUTERS_EPOCHS,
        batch_size=REUTERS_BATCH_SIZE,
        callbacks=reuters_callbacks,
        verbose=1,
    )
    train_time_s = time.perf_counter() - t0

    test_loss, test_acc = model.evaluate(X_rt_test, y_rt_test_cat, verbose=0)
    y_pred = np.argmax(model.predict(X_rt_test, verbose=0), axis=1)
    macro_f1 = f1_score(y_rt_test, y_pred, average="macro", zero_division=0)

    plot_reuters_training(history, run_id, setup)
    plot_confusion_top_classes(y_rt_test, y_pred, run_id, setup)

    row = {
        "run_id": run_id,
        "setup": setup,
        "embedding_dim": emb_dim,
        "reuters_vocab_size": REUTERS_VOCAB_SIZE,
        "coverage_pct": round(coverage_pct, 2),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(macro_f1),
        "train_time_s": round(train_time_s, 2),
    }
    append_row(REUTERS_RESULTS_PATH, row, fieldnames=REUTERS_RESULT_FIELDS)
    print(f"  test_acc={test_acc:.4f}  macro_f1={macro_f1:.4f}  coverage={coverage_pct:.1f}%")
    keras.backend.clear_session()
    return row


# Ensure experiments table exists (if §7 was skipped)
if "experiments_unique" not in globals():
    experiments_unique = pd.read_csv(EXPERIMENTS_CSV_PATH).drop_duplicates(
        subset=["run_id"], keep="last"
    )

# --- Load & preprocess Reuters ---
print("Loading Reuters dataset...")
(X_rt_train, y_rt_train), (X_rt_test, y_rt_test) = reuters.load_data(num_words=REUTERS_VOCAB_SIZE)
reuters_index_to_word, _ = build_reuters_index()

X_rt_train = pad_sequences(X_rt_train, maxlen=REUTERS_MAX_LEN, padding="post", truncating="post")
X_rt_test = pad_sequences(X_rt_test, maxlen=REUTERS_MAX_LEN, padding="post", truncating="post")

# Train/val split from training portion
X_rt_tr, X_rt_val, y_rt_tr, y_rt_val = train_test_split(
    X_rt_train,
    y_rt_train,
    test_size=VAL_FRACTION,
    random_state=RANDOM_SEED,
    stratify=y_rt_train,
)
y_rt_train_cat = to_categorical(y_rt_tr, REUTERS_NUM_CLASSES)
y_rt_val_cat = to_categorical(y_rt_val, REUTERS_NUM_CLASSES)
y_rt_test_cat = to_categorical(y_rt_test, REUTERS_NUM_CLASSES)

print(
    f"Reuters: train={len(y_rt_tr):,} val={len(y_rt_val):,} test={len(y_rt_test):,} | "
    f"max_len={REUTERS_MAX_LEN} epochs={REUTERS_EPOCHS}"
)
assert len(X_rt_tr) == len(y_rt_train_cat), "Train X/y size mismatch"
assert len(X_rt_val) == len(y_rt_val_cat), "Val X/y size mismatch"

# --- Baseline (random embeddings, dim=100) ---
BASELINE_DIM = 100
train_and_evaluate_reuters(
    "baseline_random",
    "baseline",
    embedding_matrix=None,
    trainable=True,
    coverage_pct=0.0,
)

# --- Per CBOW run: frozen + fine-tune ---
reuters_rows: list[dict] = []
for _, exp_row in experiments_unique.iterrows():
    rid = exp_row["run_id"]
    E_cbow, cbow_wi, _, _ = load_run_artifacts(rid)
    emb_matrix, coverage = build_pretrained_embedding_matrix(E_cbow, cbow_wi, reuters_index_to_word)

    for setup, trainable in [("frozen", False), ("finetune", True)]:
        row = train_and_evaluate_reuters(
            rid,
            setup,
            emb_matrix.copy(),
            trainable,
            coverage_pct=coverage,
        )
        if row:
            reuters_rows.append(row)

print("\n=== Reuters results ===")
display(pd.read_csv(REUTERS_RESULTS_PATH))

Loading Reuters dataset...
Reuters: train=8,083 val=899 test=2,246 | max_len=256 epochs=10

--- Reuters baseline | run_id=baseline_random | emb_dim=100 ---
Epoch 1/10


I0000 00:00:1779305768.102243  139930 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26528587__.31
I0000 00:00:1779305768.591705  139930 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779305768.851157  139930 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
W0000 00:00:1779305769.071465  155485 hlo_rematerialization.cc:3204] Can't reduce memory use below 9.10GiB (9775475697 bytes) by rematerialization; only reduced to 12.61GiB (13543401504 bytes), down from 12.61GiB (13543401504 bytes) originally
W0000 00:00:1779305779.156922  139930 bfc_allocator.cc:502] All

57/64 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3318 - loss: 2.9236

I0000 00:00:1779305794.163885  139931 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26528587__.31
I0000 00:00:1779305795.016626  139931 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.3400 - loss: 2.8686

...

I0000 00:00:1779306121.914643  139932 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26635247__.31


61/64 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3732 - loss: 2.7488

I0000 00:00:1779306124.121048  139932 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26635247__.31


64/64 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.4570 - loss: 2.3520 - val_accuracy: 0.5128 - val_loss: 1.9038
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5421 - loss: 1.8293 - val_accuracy: 0.5717 - val_loss: 1.7275
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5927 - loss: 1.6335 - val_accuracy: 0.5973 - val_loss: 1.6068
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6259 - loss: 1.4922 - val_accuracy: 0.6385 - val_loss: 1.5082
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6835 - loss: 1.2966 - val_accuracy: 0.6885 - val_loss: 1.3342
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7234 - loss: 1.1238 - val_accuracy: 0.7052 - val_loss: 1.2743
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7471 - loss: 0.9809 - val_accuracy: 0.7197 - val_loss: 1.2483
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7748 - loss: 0.8632 - val_accuracy: 0.7264 - val_loss: 1.

,run_id,setup,embedding_dim,reuters_vocab_size,coverage_pct,test_loss,test_accuracy,test_macro_f1,train_time_s
0,baseline_random,baseline,100,20000,0.00,1.380337,0.675423,0.111392,46.24
1,cbow_30k_win2_emb100_ep5_vocab30000,frozen,100,20000,58.90,1.476238,0.650935,0.100247,8.52
2,cbow_30k_win2_emb100_ep5_vocab30000,finetune,100,20000,58.90,1.287970,0.695904,0.116998,9.68
3,cbow_30k_win5_emb100_ep5_vocab30000,frozen,100,20000,58.90,1.501186,0.625557,0.086002,8.37
4,cbow_30k_win5_emb100_ep5_vocab30000,finetune,100,20000,58.90,1.176466,0.748887,0.231991,11.17
5,cbow_30k_win2_emb300_ep5_vocab30000,frozen,300,20000,58.90,1.398689,0.670080,0.130235,91.57
6,cbow_30k_win2_emb300_ep5_vocab30000,finetune,300,20000,58.90,1.149938,0.734639,0.193830,12.97
7,cbow_100k_win2_emb100_ep5_vocab50000,frozen,100,20000,69.67,1.473702,0.658504,0.109912,9.04
8,cbow_100k_win2_emb100_ep5_vocab50000,finetune,100,20000,69.67,1.235080,0.715494,0.151416,9.89
9,cbow_100k_win5_emb100_ep5_vocab50000,frozen,100,20000,69.67,1.451453,0.650935,0.102582,8.40


## 9.  Public Embeddings Comparison

Pre-trained **GloVe** (`glove-wiki-gigaword-100`, 100-d) mapped into the Reuters vocabulary.
Trains **frozen** and **fine-tune** classifiers; rows are appended to `reuters_results.csv`
with `run_id=public_glove-wiki-gigaword-100` and `setup=public_glove_*`.


In [11]:
import gensim.downloader as api

PUBLIC_EMBEDDING_NAME = "glove-wiki-gigaword-100"
PUBLIC_RUN_ID = f"public_{PUBLIC_EMBEDDING_NAME}"


def build_public_embedding_matrix(
    kv,
    reuters_index_to_word: dict[int, str],
    reuters_vocab_size: int = REUTERS_VOCAB_SIZE,
) -> tuple[np.ndarray, float]:
    """Map GloVe (or other KeyedVectors) into Reuters embedding matrix."""
    emb_dim = kv.vector_size
    matrix = np.random.normal(0, 0.1, (reuters_vocab_size, emb_dim)).astype(np.float32)
    found, total = 0, 0
    for rid in range(3, reuters_vocab_size):
        total += 1
        rw = reuters_index_to_word.get(rid, "")
        if not rw:
            continue
        cw = clean_sentence(rw)
        if cw in kv:
            matrix[rid] = kv[cw]
            found += 1
    coverage = 100.0 * found / total if total else 0.0
    return matrix, coverage


def ensure_reuters_data_loaded() -> None:
    """§8 must be run first; load Reuters arrays if this section runs standalone."""
    required = ("X_rt_tr", "X_rt_val", "X_rt_test", "y_rt_train_cat", "y_rt_val_cat", "y_rt_test_cat", "y_rt_test", "reuters_index_to_word")
    missing = [n for n in required if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Run §8 Reuters Classification first (missing: {missing})."
        )


ensure_reuters_data_loaded()

print(f"Loading public embeddings: {PUBLIC_EMBEDDING_NAME} ...")
glove_kv = api.load(PUBLIC_EMBEDDING_NAME)
print(f"  vectors={glove_kv.vectors.shape[0]:,}  dim={glove_kv.vector_size}")

glove_matrix, glove_coverage = build_public_embedding_matrix(glove_kv, reuters_index_to_word)
print(f"  Reuters coverage: {glove_coverage:.1f}%")

public_rows: list[dict] = []
for setup_suffix, trainable in [("frozen", False), ("finetune", True)]:
    setup = f"public_glove_{setup_suffix}"
    row = train_and_evaluate_reuters(
        PUBLIC_RUN_ID,
        setup,
        glove_matrix.copy(),
        trainable,
        coverage_pct=glove_coverage,
        skip_if_exists=True,
    )
    if row:
        public_rows.append(row)

reuters_df_public = pd.read_csv(REUTERS_RESULTS_PATH)
cbow_finetune = reuters_df_public[
    reuters_df_public["setup"].eq("finetune") & ~reuters_df_public["run_id"].str.startswith("public_")
]
best_cbow = cbow_finetune.loc[cbow_finetune["test_accuracy"].idxmax()]
public_finetune = reuters_df_public[
    (reuters_df_public["run_id"] == PUBLIC_RUN_ID)
    & reuters_df_public["setup"].str.startswith("public_glove_finetune")
]
if len(public_finetune):
    pub_acc = float(public_finetune["test_accuracy"].iloc[-1])
else:
    pub_acc = float("nan")

print("\n=== Public vs best CBOW (fine-tune) ===")
print(f"  Best CBOW:  {best_cbow['run_id']}  acc={best_cbow['test_accuracy']:.4f}")
print(f"  GloVe:      {PUBLIC_RUN_ID}  acc={pub_acc:.4f}")
if not np.isnan(pub_acc):
    delta = pub_acc - best_cbow["test_accuracy"]
    print(f"  Delta (GloVe − best CBOW): {delta:+.4f}")

Loading public embeddings: glove-wiki-gigaword-100 ...
  vectors=400,000  dim=100
  Reuters coverage: 89.0%

--- Reuters public_glove_frozen | run_id=public_glove-wiki-gigaword-100 | emb_dim=100 ---
Epoch 1/10


I0000 00:00:1779306153.094395  139928 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26642190__.29


58/64 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3974 - loss: 2.5966

I0000 00:00:1779306154.959389  139932 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26642190__.29


64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.4825 - loss: 2.1774 - val_accuracy: 0.5884 - val_loss: 1.6810
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6071 - loss: 1.6389 - val_accuracy: 0.6474 - val_loss: 1.4674
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6687 - loss: 1.4138 - val_accuracy: 0.6908 - val_loss: 1.3122
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6995 - loss: 1.2484 - val_accuracy: 0.7186 - val_loss: 1.2116
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7245 - loss: 1.1374 - val_accuracy: 0.7319 - val_loss: 1.1311
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7401 - loss: 1.0538 - val_accuracy: 0.7375 - val_loss: 1.1102
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7605 - loss: 0.9544 - val_accuracy: 0.7531 - val_loss: 1.1133
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7741 - loss: 0.8862 - val_accuracy: 0.7330 - val_loss: 1.1548
E

I0000 00:00:1779306163.912205  139928 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26648533__.31


60/64 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3576 - loss: 2.7424

I0000 00:00:1779306165.943236  139930 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26648533__.31


64/64 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.4591 - loss: 2.2955 - val_accuracy: 0.5673 - val_loss: 1.7349
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5883 - loss: 1.6954 - val_accuracy: 0.6518 - val_loss: 1.5138
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6540 - loss: 1.4566 - val_accuracy: 0.7008 - val_loss: 1.3195
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7007 - loss: 1.2657 - val_accuracy: 0.7175 - val_loss: 1.2242
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7336 - loss: 1.1168 - val_accuracy: 0.7319 - val_loss: 1.1908
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7539 - loss: 1.0039 - val_accuracy: 0.7519 - val_loss: 1.1064
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7798 - loss: 0.8942 - val_accuracy: 0.7608 - val_loss: 1.0313
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7996 - loss: 0.7925 - val_accuracy: 0.7653 - val_loss: 1.0

## 10. Results Aggregation

Load `experiments.csv`, `reuters_results.csv`, and `corpus_stats.csv`; render summary
tables and parameter-effect plots (saved under `artifacts/plots/`).

In [12]:
EXPERIMENTS_CSV_PATH = DIR_TABLES / "experiments.csv"
REUTERS_RESULTS_PATH = DIR_TABLES / "reuters_results.csv"
CORPUS_STATS_PATH = DIR_TABLES / "corpus_stats.csv"

HAND_PICKED_WORDS = ["friday", "september", "monday", "water", "president"]


def load_deduped_experiments() -> pd.DataFrame:
    if not EXPERIMENTS_CSV_PATH.exists():
        raise FileNotFoundError(f"Missing {EXPERIMENTS_CSV_PATH} — run §9 Experiment Runner first.")
    return pd.read_csv(EXPERIMENTS_CSV_PATH).drop_duplicates(subset=["run_id"], keep="last")


def load_reuters_results() -> pd.DataFrame:
    if not REUTERS_RESULTS_PATH.exists():
        raise FileNotFoundError(f"Missing {REUTERS_RESULTS_PATH} — run §8 Reuters Classification first.")
    return pd.read_csv(REUTERS_RESULTS_PATH)


def merge_experiments_reuters(exp_df: pd.DataFrame, rt_df: pd.DataFrame) -> pd.DataFrame:
    cbow_rt = rt_df[
        rt_df["run_id"].str.startswith("cbow_") & rt_df["setup"].isin(["frozen", "finetune"])
    ].copy()
    return cbow_rt.merge(exp_df, on="run_id", how="left", suffixes=("", "_exp"))


def plot_reuters_accuracy_bars(rt_df: pd.DataFrame) -> Path:
    plot_df = rt_df.copy()
    plot_df["label"] = plot_df["run_id"].str.replace("cbow_", "", regex=False).str[:28]
    plot_df.loc[plot_df["run_id"] == "baseline_random", "label"] = "baseline"
    plot_df.loc[plot_df["run_id"].str.startswith("public_"), "label"] = plot_df["run_id"].str.replace("public_", "pub:", regex=False)

    order = (
        plot_df.groupby("label")["test_accuracy"]
        .max()
        .sort_values(ascending=False)
        .index.tolist()
    )
    plt.figure(figsize=(14, 6))
    hue_order = [s for s in ["baseline", "frozen", "finetune", "public_glove_frozen", "public_glove_finetune"] if s in plot_df["setup"].unique()]
    sns.barplot(
        data=plot_df,
        x="label",
        y="test_accuracy",
        hue="setup",
        order=order,
        hue_order=hue_order,
    )
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Reuters test accuracy")
    plt.xlabel("Run")
    plt.legend(title="setup", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.title("Reuters test accuracy by embedding source and setup")
    return save_fig("reuters_test_accuracy_all_setups")


def plot_param_effect(
    merged: pd.DataFrame,
    param: str,
    title: str,
    filename: str,
    *,
    setup: str = "finetune",
    filters: dict | None = None,
) -> Path | None:
    df = merged[merged["setup"] == setup].copy()
    if filters:
        for col, val in filters.items():
            df = df[df[col] == val]
    if df[param].nunique() < 2:
        print(f"Skip {filename}: need ≥2 values for {param} (n={df[param].nunique()})")
        return None
    agg = df.groupby(param)["test_accuracy"].agg(["mean", "std", "count"]).reset_index()
    agg.columns = [param, "mean_acc", "std_acc", "n"]
    plt.figure(figsize=(7, 4))
    yerr = agg["std_acc"].fillna(0)
    plt.errorbar(agg[param], agg["mean_acc"], yerr=yerr, marker="o", capsize=4)
    for _, r in agg.iterrows():
        plt.annotate(f"n={int(r['n'])}", (r[param], r["mean_acc"]), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
    plt.xlabel(param)
    plt.ylabel(f"Reuters test accuracy ({setup})")
    plt.title(title)
    return save_fig(filename)


def plot_frozen_finetune_delta(merged: pd.DataFrame) -> Path:
    pivot = merged.pivot_table(index="run_id", columns="setup", values="test_accuracy", aggfunc="first")
    if not {"frozen", "finetune"}.issubset(pivot.columns):
        print("Skip frozen_vs_finetune_delta: missing frozen/finetune rows")
        return None
    pivot["delta"] = pivot["finetune"] - pivot["frozen"]
    pivot = pivot.sort_values("delta", ascending=True)
    plt.figure(figsize=(10, max(4, 0.35 * len(pivot))))
    colors = ["#c44e52" if d < 0 else "#55a868" for d in pivot["delta"]]
    plt.barh(pivot.index.str.replace("cbow_", "", regex=False), pivot["delta"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Δ test accuracy (fine-tune − frozen)")
    plt.title("Reuters: benefit of fine-tuning pretrained embeddings")
    return save_fig("reuters_frozen_vs_finetune_delta")


def show_cosine_tables_for_run(run_id: str, words: list[str]) -> None:
    cosine_path = DIR_TABLES / f"cosine_{run_id}.csv"
    if not cosine_path.exists():
        print(f"No cosine table: {cosine_path}")
        return
    cdf = pd.read_csv(cosine_path)
    for w in words:
        sub = cdf[(cdf["target_word"] == w)].sort_values("rank").head(10)
        if sub.empty:
            print(f"  [{w}] not in cosine table for {run_id}")
            continue
        print(f"\n### Top-10 neighbors: {w} ({run_id})")
        display(sub[["rank", "neighbor", "cosine"]])


# --- Load tables ---
exp_df = load_deduped_experiments()
rt_df = load_reuters_results()
merged_rt = merge_experiments_reuters(exp_df, rt_df)

print("=== Corpus statistics ===")
if CORPUS_STATS_PATH.exists():
    display(pd.read_csv(CORPUS_STATS_PATH))
else:
    print(f"(missing {CORPUS_STATS_PATH})")

print("\n=== CBOW experiments (deduplicated) ===")
display(exp_df.sort_values(["corpus", "window_size", "embedding_dim", "epochs"]))

print("\n=== Reuters classification results ===")
display(rt_df.sort_values(["run_id", "setup"]))

# Coverage per CBOW run (from frozen row — same matrix as finetune)
coverage_df = (
    rt_df[rt_df["run_id"].str.startswith("cbow_") & rt_df["setup"].eq("frozen")]
    .merge(exp_df[["run_id", "corpus", "vocab_size"]], on="run_id", how="left")
    [["run_id", "corpus", "vocab_size", "coverage_pct", "embedding_dim"]]
    .sort_values("coverage_pct", ascending=False)
    .rename(columns={"vocab_size": "cbow_vocab_size"})
)
print("\n=== Reuters coverage by CBOW vocabulary ===")
display(coverage_df)
coverage_df.to_csv(DIR_TABLES / "reuters_coverage_by_cbow.csv", index=False)
print(f"Saved {DIR_TABLES / 'reuters_coverage_by_cbow.csv'}")

# Best runs summary
cbow_ft = rt_df[rt_df["setup"].eq("finetune") & rt_df["run_id"].str.startswith("cbow_")]
best_cbow_row = cbow_ft.loc[cbow_ft["test_accuracy"].idxmax()]
public_ft = rt_df[rt_df["setup"].str.startswith("public_glove_finetune")]
print("\n=== Best fine-tuned Reuters accuracy ===")
display(pd.DataFrame([best_cbow_row]))
if len(public_ft):
    print("\n=== Public GloVe (fine-tune) ===")
    display(public_ft.tail(1))

# Hand-picked cosine tables (best CBOW by val_top1_acc on CBOW task)
best_qual_row = exp_df.loc[exp_df["val_top1_acc"].idxmax()]
print(f"\n=== Cosine neighbors (best CBOW val_top1: {best_qual_row['run_id']}) ===")
show_cosine_tables_for_run(best_qual_row["run_id"], HAND_PICKED_WORDS)

# --- Plots (saved + displayed) ---
plot_reuters_accuracy_bars(rt_df)
plot_frozen_finetune_delta(merged_rt)

plot_param_effect(
    merged_rt,
    "embedding_dim",
    "Effect of embedding dimension (100K, win=2, ep=5)",
    "reuters_effect_embedding_dim",
    filters={"corpus": "100k", "window_size": 2, "epochs": 5},
)
plot_param_effect(
    merged_rt,
    "window_size",
    "Effect of context window (100K, emb=100, ep=5)",
    "reuters_effect_window_size",
    filters={"corpus": "100k", "embedding_dim": 100, "epochs": 5},
)
plot_param_effect(
    merged_rt,
    "corpus",
    "Effect of corpus size (win=2, emb=100, ep=5)",
    "reuters_effect_corpus_size",
    filters={"window_size": 2, "embedding_dim": 100, "epochs": 5},
)
plot_param_effect(
    merged_rt,
    "epochs",
    "Effect of CBOW training epochs (100K, win=5, emb=100)",
    "reuters_effect_epochs",
    filters={"corpus": "100k", "window_size": 5, "embedding_dim": 100},
)

# Public vs best CBOW bar
compare_rows = []
if len(cbow_ft):
    compare_rows.append(
        {
            "source": "best_cbow_finetune",
            "run_id": best_cbow_row["run_id"],
            "test_accuracy": best_cbow_row["test_accuracy"],
        }
    )
if len(public_ft):
    compare_rows.append(
        {
            "source": "public_glove_finetune",
            "run_id": public_ft.iloc[-1]["run_id"],
            "test_accuracy": public_ft.iloc[-1]["test_accuracy"],
        }
    )
if len(rt_df[rt_df["run_id"] == "baseline_random"]):
    bl = rt_df[rt_df["run_id"] == "baseline_random"].iloc[-1]
    compare_rows.append(
        {"source": "baseline_random", "run_id": "baseline_random", "test_accuracy": bl["test_accuracy"]}
    )
if compare_rows:
    cmp_df = pd.DataFrame(compare_rows)
    plt.figure(figsize=(6, 4))
    sns.barplot(data=cmp_df, x="source", y="test_accuracy", hue="source", legend=False, palette="muted")
    plt.ylabel("Reuters test accuracy")
    plt.title("Baseline vs best CBOW vs public GloVe (fine-tune)")
    for _, r in cmp_df.iterrows():
        plt.text(
            list(cmp_df["source"]).index(r["source"]),
            r["test_accuracy"] + 0.005,
            f"{r['test_accuracy']:.3f}",
            ha="center",
            fontsize=9,
        )
    save_fig("reuters_public_vs_best_cbow")

print("\nResults aggregation complete. Figures under artifacts/plots/.")

=== Corpus statistics ===


,corpus,subset,n_sentences,avg_chars,avg_whitespace_tokens,unique_whitespace_tokens,debug
0,30k,full,30000,116.620667,19.699567,79520,False
1,30k,active_subset,30000,116.620667,19.699567,79520,False
2,100k,full,100000,116.967910,19.753540,172454,False
3,100k,active_subset,100000,116.967910,19.753540,172454,False



=== CBOW experiments (deduplicated) ===


,run_id,corpus,n_sentences,window_size,embedding_dim,vocab_size,epochs,batch_size,train_loss,val_loss,val_top1_acc,val_top5_acc,train_time_s
3,cbow_100k_win2_emb100_ep5_vocab50000,100k,99065,2,100,50000,5,256,5.141646,6.165301,0.183741,0.335743,246.45
5,cbow_100k_win2_emb300_ep5_vocab50000,100k,99065,2,300,50000,5,256,4.795797,6.152708,0.187480,0.338889,295.47
4,cbow_100k_win5_emb100_ep5_vocab50000,100k,99065,5,100,50000,5,256,5.854170,6.623945,0.137074,0.270567,150.32
6,cbow_100k_win5_emb100_ep10_vocab50000,100k,99065,5,100,50000,6,256,5.585392,6.620987,0.138319,0.271658,185.04
7,cbow_100k_win8_emb100_ep5_vocab50000,100k,99065,8,100,50000,5,256,6.295267,6.885544,0.108549,0.236890,98.03
0,cbow_30k_win2_emb100_ep5_vocab30000,30k,29687,2,100,30000,5,256,5.686319,6.477578,0.158483,0.297808,60.48
2,cbow_30k_win2_emb300_ep5_vocab30000,30k,29687,2,300,30000,5,256,5.328689,6.456617,0.162623,0.303978,52.83
1,cbow_30k_win5_emb100_ep5_vocab30000,30k,29687,5,100,30000,5,256,6.384019,6.987684,0.105866,0.233034,35.24



=== Reuters classification results ===


...


### Top-10 neighbors: water (cbow_100k_win2_emb300_ep5_vocab50000)


,rank,neighbor,cosine
100,1,pesticides,0.611768
101,2,craftsmanship,0.566066
102,3,subsystem,0.559211
103,4,scratch,0.539577
104,5,defective,0.534964
105,6,passersby,0.532669
106,7,climates,0.528082
107,8,fishers,0.508166
108,9,vacancies,0.506463
109,10,grading,0.498500



### Top-10 neighbors: president (cbow_100k_win2_emb300_ep5_vocab50000)


,rank,neighbor,cosine
500,1,nominee,0.570100
501,2,chancellor,0.561555
502,3,minister,0.556749
503,4,chairperson,0.538217
504,5,chairman,0.527143
505,6,bishop,0.520837
506,7,taoiseach,0.518725
507,8,governor,0.514466
508,9,speaker,0.514420
509,10,inspector,0.512586


Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_test_accuracy_all_setups.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_frozen_vs_finetune_delta.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_effect_embedding_dim.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_effect_window_size.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_effect_corpus_size.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_effect_epochs.png
Saved figure: /home/azaliia/projects/LM/practice/artifacts/plots/reuters_public_vs_best_cbow.png

Results aggregation complete. Figures under artifacts/plots/.


## 12. Conclusions

### Summary

In this project, we trained CBOW word embeddings on two English news corpora containing 30K and 100K sentences. In total, we tested 8 different configurations with different window sizes, embedding dimensions, and training settings.

The embeddings were evaluated in two ways:

- qualitatively, using cosine similarity and t-SNE visualizations
- quantitatively, using Reuters topic classification with a CNN classifier

We compared several initialization strategies:

- random embeddings trained from scratch
- pretrained CBOW embeddings with frozen weights
- pretrained CBOW embeddings with fine-tuning
- public GloVe embeddings (`glove-wiki-gigaword-100`)

Reuters classification was trained for up to 10 epochs with EarlyStopping based on validation metrics.

---

### Main findings

| Setup | Best result | Notes |
|---|---|---|
| Random baseline | 67.5% | Embeddings learned from scratch |
| CBOW + fine-tuning | 75.0% | `cbow_100k_win2_emb300_ep5` |
| CBOW + frozen | 67.0% | Similar to baseline |
| GloVe + fine-tuning | 75.3% | `glove-wiki-gigaword-100` |
| GloVe + frozen | 72.0% | Strong performance without tuning |

Fine-tuning consistently improved performance compared to frozen embeddings. In several experiments, the improvement was around 7–12 percentage points. For example, with the 100K corpus, window size 2, and embedding size 100, frozen embeddings achieved 65.9% accuracy, while fine-tuning reached 71.5%.

Frozen CBOW embeddings alone rarely outperformed the random baseline. One important reason is that only around 59–70% of Reuters vocabulary tokens received pretrained vectors. The remaining words stayed randomly initialized.

Using the larger 100K corpus improved downstream performance. With the same settings (`window=2`, `embedding_dim=100`, `epochs=5`), the 100K corpus achieved 71.5% accuracy after fine-tuning, compared to 69.6% for the 30K corpus.

Embedding dimensionality also had a noticeable impact. On the 100K corpus with window size 2, embeddings with dimension 300 performed better than dimension 100. The larger embeddings achieved 75.0% test accuracy, while the smaller ones reached 71.5%.

Smaller context windows worked better for Reuters classification. Window size 2 produced the strongest results, while window size 8 was weaker. Larger windows likely diluted local semantic information that is important for short-text classification.

Increasing CBOW training epochs from 5 to 10 produced only minor improvements in downstream performance. Once embeddings reached a reasonable quality level, additional CBOW training had limited impact on Reuters accuracy.

Public GloVe embeddings slightly outperformed our best CBOW embeddings. GloVe reached 75.3% test accuracy after fine-tuning, compared to 75.0% for the best CBOW model. This difference is mainly explained by better vocabulary coverage and much larger pretraining data.

---

### Qualitative evaluation

Cosine similarity and t-SNE visualizations showed meaningful semantic groupings.

Calendar-related words such as `friday`, `monday`, `thursday`, and month names formed clear clusters. Political terms such as `president`, `minister`, and `governor` were also grouped together.

Some words, especially more ambiguous ones like `water` or `music`, produced noisier neighbors. This reflects the relatively low CBOW word-prediction accuracy observed during training.

In general:

- embeddings trained on 100K performed better than those trained on 30K
- larger embedding dimensions produced more coherent semantic relationships

---

### Limitations

Several limitations should be noted:

- CBOW word-prediction accuracy remained relatively low (around 16–19% top-1 validation accuracy)
- frozen embeddings alone were usually not strong enough without downstream fine-tuning
- Reuters classes are highly imbalanced, which explains why macro-F1 scores stayed much lower than accuracy
- vocabulary alignment between Reuters, CBOW embeddings, and GloVe embeddings was imperfect, leaving some words randomly initialized
- only one downstream task and one public embedding baseline were tested
- larger experiments on the 100K corpus and higher-dimensional embeddings required significant computation time

Because of this, DEBUG mode was important for quickly validating the pipeline before running full experiments.

---

### Final takeaway

The experiments showed that CBOW embeddings trained on a news corpus can achieve competitive performance when they are fine-tuned on the downstream task.

The best results were obtained with:

- larger training data (100K corpus)
- moderate context size (`window=2`)
- higher embedding dimensionality (`d=300`)
- fine-tuning enabled during Reuters training

GloVe remained a very strong baseline because of its broader vocabulary coverage and large-scale pretraining.

Overall, the results suggest that pretrained embeddings are most useful when they are adapted to the target task, rather than used as fixed representations.